# Gemma 4 agent: L4x4 timing probe
Private measurement notebook. No submission is produced.

In [ ]:
import glob, importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

T_NOTEBOOK_START = time.time()
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()

WORKING_DIR = Path('/kaggle/working')
SUMMARY_PATH = WORKING_DIR / 'timing_summary.json'
TIMING = {'wheel_install_seconds': round(time.time() - t0, 1), 'python': sys.version.split()[0]}

def save_timing():
    TIMING['notebook_elapsed_seconds'] = round(time.time() - T_NOTEBOOK_START, 1)
    SUMMARY_PATH.write_text(json.dumps(TIMING, indent=2))

save_timing()
print(TIMING)


In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR)

(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
    'temperature: 0.2\n'
    'top_p: 0.95\n'
    'max_output_tokens: 16384\n'
    'thinking_config:\n'
    '  thinking_budget: 4096\n'
    '  include_thoughts: true\n',
    encoding='utf-8',
)
(AGENT_DIR / 'eval_config.yaml').write_text(
    'evaluation:\n'
    '  timeout_seconds: 300\n'
    '  max_tool_calls: 100\n'
    '  max_time_minutes: 5\n'
    '  max_turns: 250\n',
    encoding='utf-8',
)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
PROBE_TASK_IDS = ['fastapi_14258', 'fastapi_14479', 'fastapi_15589', 'rich_3470', 'rich_3676', 'rich_4077']
probe_tasks = [t for t in tasks if t.instance_id in PROBE_TASK_IDS]
print(len(tasks), 'tasks;', [t.instance_id for t in probe_tasks])


In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
TIMING['gpus'] = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
TIMING['tensor_parallel_size'] = tp_size

# gpu_memory_utilization 0.80 is what HARNESS_README section 3.1 states for the scorer.
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    default_chat_template_kwargs={'enable_thinking': True},
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.80,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
t0 = time.time()
server_instance.start()
TIMING['vllm_startup_seconds'] = round(time.time() - t0, 1)

try:
    log_text = Path(server_instance.log_path).read_text(errors='replace')
    TIMING['vllm_log_facts'] = [
        line.strip()[-160:] for line in log_text.splitlines()
        if any(k in line for k in ('GPU KV cache size', 'Maximum concurrency', 'Model loading took', 'Available KV cache memory'))
    ][:12]
except Exception as e:
    TIMING['vllm_log_facts'] = f'unavailable: {e}'

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key='EMPTY',
)
save_timing()
print(json.dumps(TIMING, indent=2))


In [ ]:
# ---- raw serving speed (same code as scripts/speed_probe.py in the local repo) ----
"""Measure raw serving speed of an OpenAI-compatible endpoint (stdlib only).

For several prompt sizes it reports prefill time (uncached and prefix-cached) and
decode tokens/second, plus one tool-call round trip. The same code is embedded in the
Kaggle timing notebook so local and L4x4 numbers are directly comparable.
"""


import json
import sys
import time
import urllib.request
import uuid

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'

FILLER = (
    'def resolve_dependency(graph, node, seen=None):\n'
    '    """Walk the dependency graph depth-first and return nodes in install order."""\n'
    '    seen = set() if seen is None else seen\n'
    '    order = []\n'
    '    for child in sorted(graph.get(node, ())):\n'
    '        if child not in seen:\n'
    '            seen.add(child)\n'
    '            order.extend(resolve_dependency(graph, child, seen))\n'
    '    order.append(node)\n'
    '    return order\n\n'
)


def post(base: str, path: str, body: dict, timeout: float = 900.0) -> tuple[dict, float]:
    req = urllib.request.Request(
        base.rstrip('/') + path,
        data=json.dumps(body).encode(),
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer EMPTY'},
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        out = json.loads(resp.read())
    return out, time.perf_counter() - t0


def chat(base: str, model: str, prompt: str, max_tokens: int, thinking: bool = False) -> tuple[dict, float]:
    body = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'min_tokens': max_tokens,
        'temperature': 0.0,
        'chat_template_kwargs': {'enable_thinking': thinking},
    }
    return post(base, '/chat/completions', body)


def probe(base: str = 'http://127.0.0.1:8000/v1', model: str = MODEL, decode_tokens: int = 384) -> dict:
    results: dict = {'model': model, 'sizes': []}
    chat(base, model, 'Say hi.', 8)  # warm up

    for repeats in (8, 60, 120):  # roughly 1k / 8k / 16k prompt tokens
        nonce = uuid.uuid4().hex
        prompt = f'# session {nonce}\n' + FILLER * repeats + '\nExplain this code in great detail.'
        out1, t_prefill = chat(base, model, prompt, 1)
        out_cached, t_cached = chat(base, model, prompt, 1)
        out2, t_full = chat(base, model, prompt, decode_tokens)
        usage = out2.get('usage', {})
        completion = usage.get('completion_tokens', decode_tokens)
        decode_s = max(t_full - t_cached, 1e-6)
        results['sizes'].append({
            'prompt_tokens': usage.get('prompt_tokens'),
            'prefill_seconds_uncached': round(t_prefill, 2),
            'prefill_tok_per_s_uncached': round((usage.get('prompt_tokens') or 0) / max(t_prefill, 1e-6), 1),
            'prefill_seconds_cached': round(t_cached, 2),
            'completion_tokens': completion,
            'decode_tok_per_s': round((completion - 1) / decode_s, 2),
        })

    # One realistic agent-style turn: thinking on, a tool available.
    tools = [{
        'type': 'function',
        'function': {
            'name': 'read_file',
            'description': 'Read a file from the workspace.',
            'parameters': {
                'type': 'object',
                'properties': {'filepath': {'type': 'string'}},
                'required': ['filepath'],
            },
        },
    }]
    body = {
        'model': model,
        'messages': [{'role': 'user', 'content': 'Open src/app.py so we can find the bug in the login handler.'}],
        'tools': tools,
        'tool_choice': 'auto',
        'max_tokens': 1024,
        'temperature': 0.0,
    }
    out, t = post(base, '/chat/completions', body)
    msg = out['choices'][0]['message']
    results['tool_turn'] = {
        'seconds': round(t, 2),
        'completion_tokens': out.get('usage', {}).get('completion_tokens'),
        'has_reasoning': bool(msg.get('reasoning_content') or msg.get('reasoning')),
        'tool_calls': [c['function'] for c in (msg.get('tool_calls') or [])],
        'finish_reason': out['choices'][0].get('finish_reason'),
    }
    return results



import urllib.request as _ur
with _ur.urlopen('http://127.0.0.1:8000/v1/models', timeout=30) as _resp:
    served_ids = [m['id'] for m in json.loads(_resp.read())['data']]
adapter_names = set(adapters.adapters.keys()) if adapters and getattr(adapters, 'adapters', None) else set()
base_ids = [m for m in served_ids if m not in adapter_names]
TIMING['served_model_ids'] = served_ids

TIMING['speed_base'] = probe(model=base_ids[0])
save_timing()
if adapter_names:
    lora_name = sorted(adapter_names)[0]
    TIMING['speed_lora_route'] = probe(model=lora_name)
    TIMING['speed_lora_route']['adapter'] = lora_name
save_timing()
print(json.dumps({k: TIMING[k] for k in TIMING if k.startswith('speed')}, indent=2))


In [ ]:
import asyncio
import concurrent.futures
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(fn, *args, **kwargs):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn(*args, **kwargs))).result()
    return asyncio.run(fn(*args, **kwargs))


ev = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text())['evaluation']
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=int(ev['timeout_seconds']),
    max_time_minutes=float(ev['max_time_minutes']),
    max_tool_calls=int(ev['max_tool_calls']),
    max_turns=int(ev['max_turns']),
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5
    ),
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
    display_mode='quiet',
)
evaluator = Evaluator(eval_config)

TIMING['tasks'] = []
for idx, task in enumerate(probe_tasks, start=1):
    t0 = time.time()
    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(probe_tasks))
        row = {
            'instance_id': task.instance_id,
            'wall_seconds': round(time.time() - t0, 1),
            'harness_duration_seconds': round(float(result.duration_seconds or 0), 1),
            'resolved': bool(result.resolved),
            'tool_calls': result.tool_calls,
            'llm_calls': result.total_llm_calls,
            'patch_chars': len(result.agent_patch or ''),
            'error': (result.error or '')[:200],
        }
    except Exception as e:  # keep going: timing of the remaining tasks is still useful
        row = {'instance_id': task.instance_id, 'wall_seconds': round(time.time() - t0, 1), 'exception': repr(e)[:300]}
    TIMING['tasks'].append(row)
    save_timing()
    print(row)


In [ ]:
try:
    server_instance.stop()
except Exception as e:
    print('server stop:', e)
save_timing()
print(json.dumps(TIMING, indent=2))
